In [1]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 
emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [2]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [3]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


In [4]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [5]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [6]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


-42.58225552052512


In [7]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [8]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [9]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides. 

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .] 
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .] 
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [10]:
# Initiate two matrices: 
# viterbi_value_matrix: to store the values described in the documentation above 
# viterbi_trace_matrix: to store the path the lead to the the maximum value in each cell
# For example, the first column of viterbi_trace_matrix will be 
# [0] indicating start state released `C`: even though not possible - but we just initiate
# [1] indicating Exon state released `C`:
# [2] indicating 5'ss state released `C`: even though not possible - but we just initiate
# [3] indicating Intron state released `C`: 
# [4] indicating end state released `C`: even though not possible - but we just initiate

num_states = len(states)  # 5 states
seq_len = len(query_sequence)  # length of the query sequence

# initialize matrices with -inf (log of 0)
viterbi_value_matrix = np.full((num_states, seq_len), -np.inf)
viterbi_trace_matrix = np.zeros((num_states, seq_len), dtype=int)

# fill the first column (initialization step)
# the first nucleotide is emitted from state coming from start state 's'
first_nuc = query_sequence[0]
for k in range(num_states):
    trans_prob = state_transition_prob[states['s']][k]
    emit_prob = emission_probs[k][emission_nuc_codes[first_nuc]]
    if trans_prob > 0 and emit_prob > 0:
        viterbi_value_matrix[k][0] = math.log(trans_prob) + math.log(emit_prob)
    else:
        viterbi_value_matrix[k][0] = -np.inf
    viterbi_trace_matrix[k][0] = states['s']  # all come from start state

print("First column of viterbi_value_matrix:")
for k in range(num_states):
    print(f"  State {id2state[k]}: {viterbi_value_matrix[k][0]}")


First column of viterbi_value_matrix:
  State s: -inf
  State E: -1.3862943611198906
  State 5: -inf
  State I: -inf
  State e: -inf


### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND** 

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [11]:
def calculate_prob_for_a_node(current_state, col_index):
    """
    Calculate the viterbi value for a given state at a given position.
    
    current_state: the row index (which state we are calculating for)
    col_index: the column index (which nucleotide position)
    
    Returns:
        max_val: the maximum log probability
        max_state: the index of the previous state that gave the max value
    """
    nuc = query_sequence[col_index]
    emit_prob = emission_probs[current_state][emission_nuc_codes[nuc]]
    
    # if emission probability is 0, this state can't emit this nucleotide
    if emit_prob == 0:
        return -np.inf, 0
    
    log_emit = math.log(emit_prob)
    
    max_val = -np.inf
    max_state = 0
    
    # check all previous states
    for prev_state in range(num_states):
        trans_prob = state_transition_prob[prev_state][current_state]
        
        # skip if transition is not possible or previous value is -inf
        if trans_prob == 0:
            continue
        if viterbi_value_matrix[prev_state][col_index - 1] == -np.inf:
            continue
        
        # calculate: previous viterbi value + log(transition) + log(emission)
        val = viterbi_value_matrix[prev_state][col_index - 1] + math.log(trans_prob) + log_emit
        
        if val > max_val:
            max_val = val
            max_state = prev_state
    
    return max_val, max_state


In [12]:
# Write for loops to iterate over the whole Viterbi Value matrix. 
# Each time, call the function 

# start from column 1 because column 0 is already initialized
for col in range(1, seq_len):
    for row in range(num_states):
        max_val, max_state = calculate_prob_for_a_node(row, col)
        viterbi_value_matrix[row][col] = max_val
        viterbi_trace_matrix[row][col] = max_state

# print the viterbi value matrix to check
print("Viterbi Value Matrix (showing first 5 columns):")
for row in range(num_states):
    print(f"State {id2state[row]}: {viterbi_value_matrix[row][:5]}")

print("\nViterbi Value Matrix (showing last 5 columns):")
for row in range(num_states):
    print(f"State {id2state[row]}: {viterbi_value_matrix[row][-5:]}")


Viterbi Value Matrix (showing first 5 columns):
State s: [-inf -inf -inf -inf -inf]
State E: [-1.38629436 -2.87794924 -4.36960411 -5.86125899 -7.35291387]
State 5: [        -inf         -inf         -inf         -inf -11.15957636]
State I: [-inf -inf -inf -inf -inf]
State e: [-inf -inf -inf -inf -inf]

Viterbi Value Matrix (showing last 5 columns):
State s: [-inf -inf -inf -inf -inf]
State E: [-32.71104677 -34.20270165 -35.69435653 -37.1860114  -38.67766628]
State 5: [-36.51770926 -35.06492516         -inf         -inf -42.48432877]
State I: [-32.05789888 -34.46584449 -35.48749574 -37.89544135 -38.91709259]
State e: [-inf -inf -inf -inf -inf]


In [13]:
# Write a function to trace the state path that gave the maximum probability. 
# This will be the final result. 


# HINT: You should first find the maximum value in the last column of `viterbi_value_matrix`,
# because that is the one with the largest probability. 
# The index of that value is the state of the last nucleotide.  

def traceback(viterbi_value_matrix, viterbi_trace_matrix):
    """
    Trace back through the viterbi_trace_matrix to find the best state path.
    """
    # find the state with max value in the last column
    last_col = viterbi_value_matrix[:, -1]
    best_last_state = np.argmax(last_col)
    best_prob = last_col[best_last_state]
    
    # trace back from the last column to the first
    path = [best_last_state]
    current_state = best_last_state
    
    for col in range(seq_len - 1, 0, -1):
        current_state = viterbi_trace_matrix[current_state][col]
        path.append(current_state)
    
    # reverse because we traced backward
    path.reverse()
    
    # convert state indices to state names
    state_path = ""
    for s in path:
        state_path += id2state[s]
    
    return state_path, best_prob

best_path, best_prob = traceback(viterbi_value_matrix, viterbi_trace_matrix)

print("Query Sequence:     ", query_sequence)
print("Best State Path:    ", best_path)
print("Log Probability:    ", best_prob)


Query Sequence:      CTTCATGTGAAAGCAGACGTAAGTCA
Best State Path:     EEEEEEEEEEEEEEEEEEEEEEEEEE
Log Probability:     -38.677666280562796
